# Step 0: Introduction to Nsight Compute

Nsight Compute is NVIDIA's interactive profiler for CUDA kernels, graphs and OptiX command lists. It provides detailed performance metrics, guided analysis, and optimization recommendations to help you understand and improve GPU kernel performance. The tool consists of two main components:

1. **`ncu`** — The command-line interface for collecting profiling data
2. **`ncu-ui`** — The graphical interface for analyzing and comparing reports

In this lab, we'll use both components to profile and optimize GPU kernels from Python frameworks like Numba, PyTorch, and Triton.

---

## The Command-Line Interface (`ncu`)

The `ncu` command-line tool is used to collect profiling data. It launches your application, intercepts CUDA API calls, kernel launches, and collects detailed performance metrics. Some common parameters are:

<table style="margin-left: 0;">
    <tr><th>Parameter</th><th>Description</th></tr>
    <tr><td><code>--set full</code></td><td>Collect the full set of curated metrics for maximum detail. This gives us the best level of insight, and since profiling overhead isn't a concern for this tutorial, we'll use it throughout.</td></tr>
    <tr><td><code>-c N</code></td><td>Collect N kernel launches. We typically use <code>-c 1</code> to profile just one invocation of our target kernel.</td></tr>
    <tr><td><code>-k regex:pattern</code></td><td>Filter kernels by name using a regex pattern. This is essential when your application launches many kernels but you only care about specific ones.</td></tr>
    <tr><td><code>--import-source yes</code></td><td>Embed source code directly in the report. This ensures the report always refers to the original source from when it was collected, even if you modify the code later.</td></tr>
    <tr><td><code>-f -o file</code></td><td>Write the report to a file, forcing overwrite if it exists. You can then open this report in the Nsight Compute UI for interactive analysis.</td></tr>
</table>

A typical profiling command looks like:

```bash
ncu --set full -c 1 -k regex:my_kernel --import-source yes -f -o report python3 my_script.py
```

This profiles the first launch of any kernel matching `my_kernel`, collecting full metrics and embedding the source code in the report file. For a full list of available options, see the documentation or use `ncu --help`.

### Console Output Options

In some environments, you may not want or be able to open the report in the UI. Fortunately, `ncu` can print profiling results directly to the console (and does so by default, if no output report is specified):

<table style="margin-left: 0;">
    <tr><th>Option</th><th>Description</th></tr>
    <tr><td><code>--page details</code></td><td>Show the "Details" page command line equivalent</td></tr>
    <tr><td><code>--page source --print-source cuda,sass</code></td><td>Show metrics associated with CUDA+SASS lines</td></tr>
    <tr><td><code>--print-summary per-kernel</code></td><td>Print a per-kernel summary across all profiled results</td></tr>
    <tr><td><code>--print-details all</code></td><td>Show more section metric details on the console</td></tr>
    <tr><td><code>--csv</code></td><td>Output in CSV format for scripting and automation</td></tr>
    <tr><td><code>--print-units base</code></td><td>Display metrics in their base units</td></tr>
    <tr><td><code>...</code></td><td>...</td></tr>
</table>

---

## The Nsight Compute UI

While the command-line interface is powerful for scripting and quick checks, the Nsight Compute UI provides a much richer interactive analysis experience. The UI presents the report as a series of *pages*, each providing a different view of the same profiling data.

### Summary Page

Opening a report file starts on the **Summary** page. It shows:
- A table of all collected results (kernel launches) in the report
- **Prioritized Rules** — The most important optimization suggestions, ordered by estimated speedup potential. The prioritized rules are implemented in Python and analyze the collected metrics to suggest optimizations. You can dive into promising optimizations directly by following the links, or switch to other pages for a more detailed view.

<img src="images/ncu_ui_1_summary.png" width="75%"/>


### Details Page

The **Details** page shows detailed statistics grouped into logical *sections*. Reading sections from top (high-level) to bottom (low-level) gives a good understanding of performance characteristics.
Each section can be expanded or collapsed using the triangle icon. Some sections have multiple views available via a dropdown in the section header.

<img src="images/ncu_ui_1_details_1.png" width="75%"/>

**Key sections include:**

<table style="margin-left: 0;">
    <tr><th>Section</th><th>What It Shows</th></tr>
    <tr><td><b>GPU Speed of Light</b></td><td>High-level compute vs memory throughput. Shows whether your kernel is compute-bound or memory-bound.</td></tr>
    <tr><td><b>Compute Workload Analysis</b></td><td>Instruction mix and pipeline utilization. Shows which hardware units (FP32, FP64, tensor cores) are being used.</td></tr>
    <tr><td><b>Memory Workload Analysis</b></td><td>Memory hierarchy utilization including L1/L2 cache hit rates and bandwidth.</td></tr>
    <tr><td><b>Warp State Statistics</b></td><td>Thread execution patterns and stall reasons. Helps identify what threads are waiting for.</td></tr>
    <tr><td><b>Occupancy</b></td><td>SM resource utilization showing how many warps can run concurrently.</td></tr>
</table>

For a comprehensive list of available sections, use the `ncu --list-sections` command or check [online here](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#sections-and-rules).

The **GPU Speed of Light** section is particularly important — it immediately tells you whether your kernel is limited by compute or memory. You can switch this section to show a different body, the *GPU Throughput Breakdown*, which shows each sub-pipeline of compute and memory to identify which one causes the overall utilization:

<img src="images/ncu_ui_1_details_2.png" width="75%"/>

The **Memory Chart** in the **Memory Workload Analysis** section visualizes the memory hierarchy. It can be switched to a very detailed table format using the section header.

<img src="images/ncu_ui_1_memory_chart.png" width="75%"/>

### Source Page

The Source page correlates assembly (SASS) with high-level code such as CUDA-C, Python or PTX.
In addition, it displays instruction-correlated metrics to help pinpoint performance problems in your code.
It's also available for viewing cubins extracted from binaries or comparing results.

<img src="images/ncu_ui_1_source_page.png" width="75%"/>

---

## The Python Report Interface

The [Python Report Interface](https://docs.nvidia.com/nsight-compute/PythonReportInterface/index.html) is a Python API for interacting with ncu report files.

The module is called `ncu_report` and works on any recent Python version. It can be found in the `extras/python` directory of your Nsight Compute installation, and on [PyPI](https://pypi.org/project/ncu-report/).

You can find examples on [https://github.com/NVIDIA/nsight-training](https://github.com/NVIDIA/nsight-training/tree/master/cuda/nsight_compute/python_report_interface).

---

## Summary

<div class="alert alert-block alert-success">
    <p><b>ncu</b>: Use <code>--set full</code> for comprehensive metrics, <code>-k regex:name</code> to filter kernels, <code>--import-source yes</code> for portable reports.</p>
    <p><b>Summary Page</b>: Start with prioritized rules to find the highest-impact optimizations.</p>
    <p><b>Details Page</b>: GPU Speed of Light shows compute vs memory vs latency bound. Sections support structured performance triage. Follow rule advise and links.</p>
    <p><b>Source Page</b>: Shows metrics per line/instructions. Correlates source with assembly. Analysis tables aggregate insights.</p>
    <p><b>Python Report Interface</b>: Python module for loading and analyzing ncu report files. <code>import ncu_report</code></p>
</div>

**Next: [Step 1 - Profiling Numba CUDA Kernels](step1.ipynb)**